### Data Collector Notebook

This notebook contains code to export data from the Yahoo Finance API to .csv files.

In [ ]:
import yfinance as yf
from datetime import date, timedelta
from pandas import DataFrame
import numpy as np
import os
import stockdex as sd

DATE_LEN = 10


In [ ]:
def save_phist_for_symbols(syms: list[str], start_date: date, end_date: date, interval: str = "1d", file='raw_phist.csv'):
    phistory = yf.download(syms, start=start_date, end=end_date, interval=interval, auto_adjust=True)

    if os.path.exists(file): os.remove(file)

    phistory = phistory.drop(labels=["High", "Low", "Open"], axis=1)

    phistory = phistory.dropna(axis=1, how="all")

    phistory.to_csv(file)

In [ ]:
def parse_symbol_list(file: str, max: int = 5000, skip_header: bool = True, exclude_fishy: bool = True) -> list[str]:
    output = []

    def is_fishy(sym: str) -> bool:
        return sym.find('.') != -1

    with open(file) as f:
        if skip_header: f.readline()
        count = 0
        while (line := f.readline()) != '' and count < max:
            sym = line.split('\t')[0]
            if not is_fishy(sym) or not exclude_fishy:
                output.append(sym)
            count += 1

    return output


In [ ]:
def save_macro_trends_for_symbols(syms: list[str], start_date: date, end_date: date):
    for sym in syms:
        ticker = sd.Ticker(ticker=sym)
        #income_st = ticker.macrotrends_income_statement(frequency="quarterly")

        #print(income_st)

        #kfrs = ticker.macrotrends_key_financial_ratios.dropna(axis=0, how='any')

        #print(kfrs)

        ebitas = ticker.macrotrends_ebitda_margin.drop(['TTM Revenue', 'TTM EBITDA'], axis=1)

        earnings = ticker.finviz_earnings_data().drop(['fiscalPeriod', 'earningsDate', 'salesActual', 'epsReportedActual', 'epsReportedEstimate', 'salesEstimate', 'epsReportedAnalysts', 'salesAnalysts'], axis=1)

        margin = ticker.macrotrends_operating_margin.drop(['TTM Revenue', 'TTM Operating Income'], axis=1).dropna(axis=0, how='all')

        #print(ebitas)

        print(earnings)

        #print(margin)

In [ ]:
save_macro_trends_for_symbols(['AAPL'], None, None)

In [ ]:
# Test Save is working on a single ticker
#save_phist_for_symbol("AAPL", date(2000, 1, 1), date(2026,1,1))

In [ ]:
symbols = parse_symbol_list("Symbols_TSX.txt")
print("Loading", len(symbols), "symbols...")

In [ ]:
symbols = []

with open('s&p500.csv') as file:
    count = 0
    while (line := file.readline()) != '' and count < 510:
        sym = line.split(',')[0]
        symbols.append(sym)
        count += 1

len(symbols)

In [ ]:
save_phist_for_symbols(symbols, date(1980, 1, 1), date(2026, 4, 1))

In [ ]:
SP500 = "^GSPC"

phistory = yf.download(SP500, start=date(1980, 1, 1), end=date(2026, 4, 1), interval="1d", auto_adjust=True)

if os.path.exists("s&p_phist.csv"): os.remove("s&p_phist.csv")

phistory = phistory.drop(labels=["High", "Low", "Open"], axis=1)

phistory = phistory.dropna(axis=1, how="all")

phistory.to_csv("s&p_phist.csv")

In [ ]:
symbols = []

with open("TSX_60.txt") as f:
    while (line := f.readline()) != '':
        sym = line.split('-')[0]
        symbols.append(sym)

save_phist_for_symbols(symbols, date(1980, 1, 1), date(2026, 4, 1), file='raw_tsx60.csv')